Import and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the parquet file built by fetch_data.py
df = pd.read_parquet('../data/tracks.parquet')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()

# Quick peek at the data
df.head(10)

Check for missing values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print(f'Valence  — min: {df.valence.min():.3f}  max: {df.valence.max():.3f}  mean: {df.valence.mean():.3f}')
print(f'Energy   — min: {df.energy.min():.3f}  max: {df.energy.max():.3f}  mean: {df.energy.mean():.3f}')
print()

# These must pass before you continue
# If they fail, check the normalisation step in fetch_data.py
assert df.valence.between(0, 1).all(), 'FAIL: valence out of 0-1 range'
assert df.energy.between(0, 1).all(),  'FAIL: energy out of 0-1 range'
assert df['id'].nunique() == len(df),  'FAIL: duplicate IDs found'
print('All checks passed. Dataset is clean.')

Plot the mood space

In [ ]:

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(df.valence, df.energy, alpha=0.09, s=5, color='#7C3AED')

# Four emotional quadrants with Hindi labels
quadrants = [
    (0.04, 0.95, 'Gussa / Angry',    '#991B1B'),
    (0.72, 0.95, 'Josh / Excited',    '#065F46'),
    (0.04, 0.04, 'Udaas / Sad',       '#1E3A5F'),
    (0.70, 0.04, 'Sukoon / Peaceful', '#78350F'),
    (0.38, 0.52, 'Romantic',          '#6D28D9'),
]
for x, y, label, color in quadrants:
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=10, color=color, va='top', fontweight='bold')

ax.axvline(0.5, color='#E5E7EB', linewidth=0.8, linestyle='--')
ax.axhline(0.5, color='#E5E7EB', linewidth=0.8, linestyle='--')
ax.set_xlabel('Valence  (Udaas → Khushi)', fontsize=12)
ax.set_ylabel('Energy   (Sukoon → Josh)',   fontsize=12)
ax.set_title(f'Indian/Bollywood track distribution in mood space  ({len(df):,} tracks)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/mood_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/mood_space.png — add this image to your README')

correlation heatmap of audio features

In [ ]:
features = [f for f in ['valence', 'energy', 'danceability', 'tempo', 'acousticness']
            if f in df.columns]

corr = df[features].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Audio feature correlations — Indian music dataset')
plt.tight_layout()
plt.show()

val_nrg_corr = corr.loc['valence', 'energy']
print(f'Valence-Energy correlation: {val_nrg_corr:.3f}')
if val_nrg_corr < 0.5:
    print('Good — valence and energy are independent enough for a 2D mood space')
else:
    print('Warning — high correlation. Consider using more diverse playlists.')

Sample a journey manually — quick mental test

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

coords = df[['valence', 'energy']].values

def find_tracks_near(valence, energy, n=5):
    target = np.array([[valence, energy]])
    dists  = euclidean_distances(coords, target).flatten()
    idx    = dists.argsort()[:n]
    return df.iloc[idx][['name', 'artist', 'valence', 'energy']]

print('Tracks closest to UDAAS (sad + calm) — val=0.15, nrg=0.20:')
print(find_tracks_near(0.15, 0.20).to_string())
print()

print('Tracks closest to JOSH (excited + energetic) — val=0.80, nrg=0.85:')
print(find_tracks_near(0.80, 0.85).to_string())
print()

print('Tracks closest to ROMANTIC — val=0.75, nrg=0.45:')
print(find_tracks_near(0.75, 0.45).to_string())
print()

print('Tracks closest to SUKOON (peaceful) — val=0.65, nrg=0.20:')
print(find_tracks_near(0.65, 0.20).to_string())